In [ ]:
pip install earthaccess rioxarray rasterio whitebox numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.2 MB/s  0:00:00
  Attempting uninstall: xarray0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/8 [whitebox]
    Found existing installation: xarray 2025.12.0━━━━━━━━━━━━━ 2/8 [whitebox]
    Uninstalling xarray-2025.12.0:╺━━━━━━━━━━━━━━ 5/8 [xarray]
      Successfully uninstalled xarray-2025.12.0━━━━━━━━━━━━━━ 5/8 [xarray]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [earthaccess] [rioxarray]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import earthaccess
from datetime import date, timedelta

# Autenticazione (al primo avvio chiede username e password di NASA Earthdata)
earthaccess.login()

# Definisci l'area di interesse (es. zona agricola in Piemonte)
bbox = (11.55, 43.08, 11.82, 43.26)

# 1. Scarica elevazione (NASADEM a 30m)
dem_results = earthaccess.search_data(short_name="NASADEM_HGT",bounding_box=bbox)
dem_files = earthaccess.download(dem_results, "./dati/dem")

# 2. Scarica immagini multispettrali HLS Sentinel-2 (HLSS30 a 30m)
oggi = date.today()
data_inizio = oggi - timedelta(days=45)
intervallo_dinamico = (data_inizio.strftime("%Y-%m-%d"), oggi.strftime("%Y-%m-%d"))
hls_results = earthaccess.search_data(short_name="HLSS30",bounding_box=bbox,temporal=intervallo_dinamico,cloud_cover=(0, 15))
hls_files = earthaccess.download(hls_results[:1], "./dati/hls")

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/18 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/18 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
import glob
import os
import zipfile
import numpy as np
import rioxarray as rxr
import whitebox

dem_dir = os.path.abspath("./dati/dem")
out_dir = os.path.abspath("./output")
os.makedirs(out_dir, exist_ok=True)

# 0. Estrai eventuali archivi .zip e trova i file .hgt
for zip_path in glob.glob(os.path.join(dem_dir, "*.zip")):
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dem_dir)

hgt_files = glob.glob(os.path.join(dem_dir, "*.hgt"))
if not hgt_files:
    raise FileNotFoundError(
        f"Nessun file .hgt trovato in {dem_dir}. "
        "Esegui prima la cella con earthaccess.download() per scaricare il DEM."
    )

print("File DEM trovato:", hgt_files[0])

dem_utm_path = os.path.join(out_dir, "dem_utm.tif")
dem_filled_path = os.path.join(out_dir, "dem_filled.tif")
slope_path = os.path.join(out_dir, "slope_deg.tif")
aspect_path = os.path.join(out_dir, "aspect_deg.tif")
sca_path = os.path.join(out_dir, "sca.tif")
twi_path = os.path.join(out_dir, "twi.tif")

# 1. Carica il DEM, convertilo in float32 e riproiettalo in UTM (metri)
dem_raw = rxr.open_rasterio(hgt_files[0]).squeeze().astype("float32")
dem_raw = dem_raw.where(dem_raw != -32768)  # Converte il valore NoData SRTM in NaN
utm_crs = dem_raw.rio.estimate_utm_crs()
dem_utm = dem_raw.rio.reproject(utm_crs)
dem_utm.rio.to_raster(dem_utm_path)

# 2. Inizializza WhiteboxTools con percorsi assoluti e log attivi
wbt = whitebox.WhiteboxTools()
wbt.set_working_dir(out_dir)
wbt.verbose = True

wbt.fill_depressions(dem=dem_utm_path, output=dem_filled_path)
wbt.slope(dem=dem_filled_path, output=slope_path, units="degrees")
wbt.aspect(dem=dem_filled_path, output=aspect_path)
wbt.d_inf_flow_accumulation(
    i=dem_filled_path,
    output=sca_path,
    out_type="Specific Contributing Area"
)

# Verifica che WhiteboxTools abbia scritto i file
if not os.path.exists(slope_path):
    raise RuntimeError(
        "WhiteboxTools non ha generato slope_deg.tif. "
        "Controlla i messaggi stampati sopra nel terminale."
    )

# 3. Calcola il TWI
slope_da = rxr.open_rasterio(slope_path).squeeze()
sca_da = rxr.open_rasterio(sca_path).squeeze()

slope_rad = np.radians(slope_da.values)
tan_slope = np.maximum(np.tan(slope_rad), 0.001)
sca_safe = np.maximum(sca_da.values, 1.0)

twi_array = np.log(sca_safe / tan_slope)
twi_da = slope_da.copy(data=twi_array)
twi_da.rio.to_raster(twi_path)

print("File generati con successo:", os.listdir(out_dir))

File DEM trovato: /kaggle/working/dati/dem/n43e011.hgt
Decompressing WhiteboxTools_linux_amd64.zip ...
WhiteboxTools package directory: /usr/local/lib/python3.13/dist-packages/whitebox
./whitebox_tools --run="FillDepressions" --wd="/kaggle/working/output" --dem='/kaggle/working/output/dem_utm.tif' --output='/kaggle/working/output/dem_filled.tif' --fix_flats -v --compress_rasters=False

******************************
* Welcome to FillDepressions *
* Powered by WhiteboxTools   *
* www.whiteboxgeo.com        *
******************************
Reading data...
Finding pit cells: 25%
Finding pit cells: 50%
Finding pit cells: 75%
Finding pit cells: 100%
Filling depressions: 0%
Filling depressions: 1%
Filling depressions: 2%
Filling depressions: 3%
Filling depressions: 4%
Filling depressions: 5%
Filling depressions: 6%
Filling depressions: 7%
Filling depressions: 8%
Filling depressions: 9%
Filling depressions: 10%
Filling depressions: 11%
Filling depressions: 12%
Filling depressions: 13%
Filling

In [ ]:
import glob

def load_and_align_band(pattern, reference_raster):
    file_path = glob.glob(pattern)[0]
    band = rxr.open_rasterio(file_path).squeeze()
    # Filtra i valori NoData (-9999) e applica il fattore di scala NASA (0.0001)
    band = band.where(band != -9999) * 0.0001
    # Allinea risoluzione, estensione e proiezione a quella del DEM UTM
    return band.rio.reproject_match(reference_raster)

# Carica le bande da HLSS30 allineandole al DEM
blue  = load_and_align_band("./dati/hls/*B02.tif", dem_utm)
red   = load_and_align_band("./dati/hls/*B04.tif", dem_utm)
nir   = load_and_align_band("./dati/hls/*B8A.tif", dem_utm)
swir1 = load_and_align_band("./dati/hls/*B11.tif", dem_utm)

# Calcolo NDVI (-1 a +1: vigore vegetativo)
ndvi = (nir - red) / (nir + red + 1e-6)

# Calcolo Bare Soil Index (-1 a +1: valori alti = suolo nudo esposto)
bsi = ((swir1 + red) - (nir + blue)) / ((swir1 + red) + (nir + blue) + 1e-6)

ndvi.rio.to_raster("./output/ndvi.tif")
bsi.rio.to_raster("./output/bsi.tif")

In [ ]:
import pandas as pd

aspect_da = rxr.open_rasterio("./output/aspect_deg.tif").squeeze()

# Crea le griglie delle coordinate X e Y in metri
xx, yy = np.meshgrid(dem_utm.x.values, dem_utm.y.values)

df_terreno = pd.DataFrame({
    "x_utm": xx.ravel(),
    "y_utm": yy.ravel(),
    "elevazione_m": dem_utm.values.ravel(),
    "slope_deg": slope_da.values.ravel(),
    "aspect_deg": aspect_da.values.ravel(),
    "twi": twi_da.values.ravel(),
    "ndvi": ndvi.values.ravel(),
    "bsi": bsi.values.ravel()
}).dropna()

print(df_terreno.head())

           x_utm         y_utm  elevazione_m  slope_deg  aspect_deg       twi  \
0  660351.398217  4.876251e+06      -32768.0   -32768.0    -32768.0  6.907755   
1  660378.392953  4.876251e+06      -32768.0   -32768.0    -32768.0  6.907755   
2  660405.387689  4.876251e+06      -32768.0   -32768.0    -32768.0  6.907755   
3  660432.382425  4.876251e+06      -32768.0   -32768.0    -32768.0  6.907755   
4  660459.377161  4.876251e+06      -32768.0   -32768.0    -32768.0  6.907755   

   ndvi  bsi  
0  -0.0 -0.0  
1  -0.0 -0.0  
2  -0.0 -0.0  
3  -0.0 -0.0  
4  -0.0 -0.0  


In [ ]:
df_crops = pd.read_csv('/kaggle/input/datasets/andreatrexx/dataset-crops/dataset_crops.csv')
df_crops.head()

,crop_code,crop_name,category,raster_path,raster_available
0,ALF,Alfalfa,Fodder crops,gs://fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPS...,True
1,AVOC,Avocado,Fruits,gs://fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPS...,True
2,AVOST,Avocado subtropical ecotype,Fruits,gs://fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPS...,True
3,AVOTH,Avocado tropical highland,Fruits,gs://fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPS...,True
4,AVOTL,Avocado tropical lowland,Fruits,gs://fao-gismgr-gaez-v5-data/DATA/GAEZ-V5/MAPS...,True


In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
import requests


def _stima_utm_epsg(bbox: tuple) -> str:
    """Stima il codice EPSG UTM locale a partire dal centro della bbox (lon_min, lat_min, lon_max, lat_max)."""
    lon_c = (bbox[0] + bbox[2]) / 2.0
    lat_c = (bbox[1] + bbox[3]) / 2.0
    zona = int((lon_c + 180) // 6) + 1
    return f"EPSG:{32600 + zona if lat_c >= 0 else 32700 + zona}"


def descrivi_terreno_zona(
    df_terreno: pd.DataFrame,
    bbox: tuple,
    utm_epsg: str = None,
) -> tuple[str, pd.DataFrame]:
    """
    Filtra df_terreno sulla bbox (lon_min, lat_min, lon_max, lat_max)
    e genera una breve descrizione del terreno usando le soglie agronomiche.
    """
    if utm_epsg is None:
        utm_epsg = _stima_utm_epsg(bbox)

    x_min, y_min, x_max, y_max = transform_bounds("EPSG:4326", utm_epsg, *bbox)

    zona_df = df_terreno[
        (df_terreno["x_utm"] >= x_min) & (df_terreno["x_utm"] <= x_max)
        & (df_terreno["y_utm"] >= y_min) & (df_terreno["y_utm"] <= y_max)
    ].copy()

    if zona_df.empty:
        return (
            "No pixels from df_terreno fall within the selected geographic area. "
            "Please verify that the bbox is inside the area downloaded from SRTM/HLS.",
            zona_df,
        )

    ettari = (len(zona_df) * 900) / 10_000.0
    slope_m = zona_df["slope_deg"].mean()
    twi_m = zona_df["twi"].mean()
    elev_m = zona_df["elevazione_m"].mean()
    ndvi_m = zona_df["ndvi"].mean()

    esposto_sud = (zona_df["aspect_deg"] >= 135.0) & (zona_df["aspect_deg"] <= 225.0)
    pct_erosione_alta = ((zona_df["slope_deg"] >= 10.0) & (zona_df["bsi"] > 0.12)).mean() * 100
    pct_erosione_mod = (zona_df["slope_deg"] >= 8.0).mean() * 100
    pct_ristagno = ((zona_df["twi"] >= 12.0) & (zona_df["slope_deg"] < 4.5)).mean() * 100
    pct_siccita = (
        (zona_df["twi"] <= 5.5) | ((zona_df["twi"] <= 6.8) & esposto_sud & (zona_df["slope_deg"] >= 3.0))
    ).mean() * 100
    pct_suolo_nudo = ((zona_df["bsi"] > 0.15) & (zona_df["ndvi"] < 0.20)).mean() * 100

    caratteristiche = []
    if pct_erosione_alta > 10:
        caratteristiche.append(
            f"high erosion risk ({pct_erosione_alta:.0f}% of the area is on a slope >= 10° with bare soil)"
        )
    elif pct_erosione_mod > 15:
        caratteristiche.append(
            f"hilly morphology with significant slopes ({pct_erosione_mod:.0f}% over 8°)"
        )

    if pct_ristagno > 10:
        caratteristiche.append(
            f"tendency toward waterlogging in depressions/valley floors ({pct_ristagno:.0f}% with TWI >= 12)"
        )

    if pct_siccita > 15:
        caratteristiche.append(
            f"rapid drainage and vulnerability to summer drought ({pct_siccita:.0f}% with low TWI or south-facing slopes)"
        )

    if pct_suolo_nudo > 20:
        caratteristiche.append(
            f"extensive presence of bare/plowed soil ({pct_suolo_nudo:.0f}% with BSI > 0.15)"
        )

    if elev_m >= 500:
        caratteristiche.append(
            f"high average elevation ({elev_m:.0f} m a.s.l.) with a shorter growing season"
        )

    if not caratteristiche:
        sintesi_criticita = (
            "Morphologically balanced terrain, with gentle slopes, good drainage "
            "and no significant physical limitations."
        )
    else:
        sintesi_criticita ="Key limitations: " + "; ".join(caratteristiche) + "."

    descrizione = (
        f"Analyzed area: {ettari:.1f} ha (average elevation {elev_m:.0f} m a.s.l., "
        f"average slope {slope_m:.1f}°, average TWI {twi_m:.1f}, average NDVI {ndvi_m:.2f}). "
        f"{sintesi_criticita}"
    )
    return descrizione, zona_df


def _scarica_raster_gaez(row_tuple, cache_dir: str) -> tuple[str, str]:
    """Converte gs:// in https:// e scarica il GeoTIFF nella cache locale se non presente."""
    crop_code, gs_path = row_tuple
    https_url = gs_path.replace("gs://", "https://storage.googleapis.com/", 1)
    local_path = os.path.join(cache_dir, f"{crop_code}.tif")

    if not os.path.exists(local_path):
        try:
            r = requests.get(https_url, timeout=30)
            if r.status_code == 200:
                with open(local_path, "wb") as f:
                    f.write(r.content)
            else:
                return crop_code, ""
        except Exception:
            return crop_code, ""
    return crop_code, local_path


def trova_coltura_piu_adattabile(
    df_crops: pd.DataFrame,
    bbox: tuple,
    cache_dir: str = "./gaez_cache",
    max_workers: int = 8,
) -> pd.DataFrame:
    """
    Scarica i raster di adattabilità da dataset_crops.csv, calcola l'indice medio
    climatico nella bbox e restituisce il DataFrame ordinato.
    """
    os.makedirs(cache_dir, exist_ok=True)
    disponibili = df_crops[df_crops["raster_available"] == True].copy()

    tasks = list(zip(disponibili["crop_code"], disponibili["raster_path"]))
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        risultati_dl = dict(executor.map(lambda t: _scarica_raster_gaez(t, cache_dir), tasks))

    lon_c = (bbox[0] + bbox[2]) / 2.0
    lat_c = (bbox[1] + bbox[3]) / 2.0
    punteggi = []

    for _, row in disponibili.iterrows():
        tif_path = risultati_dl.get(row["crop_code"], "")
        if not tif_path or not os.path.exists(tif_path):
            punteggi.append(np.nan)
            continue

        with rasterio.open(tif_path) as src:
            window = from_bounds(*bbox, transform=src.transform)
            data = src.read(1, window=window)

            if data.size == 0:
                val = list(src.sample([(lon_c, lat_c)]))[0][0]
                data = np.array([val])

            valid = data[(data >= 0) & (data <= 10000)]
            if src.nodata is not None:
                valid = valid[valid != src.nodata]

            score_pct = float(valid.mean() / 100.0) if valid.size > 0 else np.nan
            punteggi.append(round(score_pct, 1) if not np.isnan(score_pct) else np.nan)

    disponibili["adattabilita_pct"] = punteggi
    classifica = (
        disponibili.dropna(subset=["adattabilita_pct"])
        .sort_values(by="adattabilita_pct", ascending=False)
        .reset_index(drop=True)
    )
    return classifica


def classifica_colture_con_terreno(
    classifica_gaez: pd.DataFrame,
    zona_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Combina l'adattabilità climatica FAO GAEZ con le criticità fisiche locali
    di df_terreno (erosione, ristagno, siccità) per ricalcolare la classifica.
    """
    df_out = classifica_gaez.copy()
    df_out["punteggio_finale"] = df_out["adattabilita_pct"]
    df_out["note_agronomiche"] = "Standard climatic adaptability"

    if zona_df.empty or df_out.empty:
        return df_out

    pct_erosione = (
        (zona_df["slope_deg"] >= 8.0)
        | ((zona_df["slope_deg"] >= 6.0) & (zona_df["bsi"] > 0.12))
    ).mean()
    pct_ristagno = ((zona_df["twi"] >= 12.0) & (zona_df["slope_deg"] < 4.5)).mean()
    pct_siccita = (zona_df["twi"] <= 6.0).mean()

    colture_sarchiate_erosive = {"MZE", "MZS", "SFL", "SUB", "SOY", "TOB", "WPO", "COT"}
    colture_protettive = {"ALF", "PST", "BRL", "WHE", "RYE", "OAT", "TRI"}
    colture_sensibili_ristagno = {"ALF", "CHK", "WPO", "OLV"}
    colture_idroesigenti = {"MZE", "MZS", "SOY", "SUB", "RCW"}
    colture_xerofile = {"SRG", "CHK", "BRL", "OLV", "SFL"}

    for idx, row in df_out.iterrows():
        code = row["crop_code"]
        score = row["punteggio_finale"]
        note = []

        if pct_erosione > 0.20:
            if code in colture_sarchiate_erosive:
                score -= 25.0 * pct_erosione
                note.append("penalized for erosion risk on slopes")
            elif code in colture_protettive:
                score += 10.0 * pct_erosione
                note.append("favored for soil protection")

        if pct_ristagno > 0.15 and code in colture_sensibili_ristagno:
            score -= 30.0 * pct_ristagno
            note.append("penalized for waterlogging/root asphyxiation risk")

        if pct_siccita > 0.20:
            if code in colture_idroesigenti:
                score -= 20.0 * pct_siccita
                note.append("penalized for high water demand")
            elif code in colture_xerofile:
                score += 10.0 * pct_siccita
                note.append("favored for drought resistance")

        df_out.at[idx, "punteggio_finale"] = round(float(np.clip(score, 0, 100)), 1)
        if note:
            df_out.at[idx, "note_agronomiche"] = "; ".join(note).capitalize()

    return df_out.sort_values(by="punteggio_finale", ascending=False).reset_index(drop=True)


def analizza_zona_geografica(
    df_terreno: pd.DataFrame,
    df_crops: pd.DataFrame,
    bbox: tuple,
    utm_epsg: str = None,
) -> pd.DataFrame:
    """
    Stampa la descrizione del terreno della zona e restituisce un DataFrame pandas
    con TUTTE le colture ordinate dalla più adattabile alla meno adattabile.
    """
    descrizione, zona_df = descrivi_terreno_zona(df_terreno, bbox, utm_epsg=utm_epsg)
    classifica_base = trova_coltura_piu_adattabile(df_crops, bbox)
    classifica_colture = classifica_colture_con_terreno(classifica_base, zona_df)

    if classifica_colture.empty:
        raise RuntimeError("Nessun dato GAEZ valido scaricato. Verifica che Internet sia attivo su Kaggle.")

    # Aggiunge il numero di posizione in classifica (1, 2, ..., 77)
    classifica_colture.insert(0, "posizione", range(1, len(classifica_colture) + 1))

    # Salva la descrizione del terreno nei metadati del DataFrame per recuperarla se serve
    classifica_colture.attrs["descrizione_terreno"] = descrizione

    print("=== TERRAIN DESCRIPTION ===")
    print(descrizione)

    return classifica_colture

Prima si estrae il potenziale climatico macro-regionale dai raster FAO GAEZ (adattabilita_pct), poi si sintetizzano i pixel topografici del campo in quote percentuali di rischio, e infine si applicano bonus e penalità agronomiche proporzionali alle caratteristiche fisiologiche di ciascuna coltura (punteggio_finale)

In [ ]:
# Asciano
bbox_utente = (11.46, 43.14, 11.66, 43.28)
risultato_asciano = analizza_zona_geografica(df_terreno, df_crops, bbox=bbox_utente)
display(risultato_asciano[["posizione", "crop_code", "crop_name", "category", "adattabilita_pct", "punteggio_finale", "note_agronomiche"]])

=== TERRAIN DESCRIPTION ===
Analyzed area: 21109.9 ha (average elevation 292 m a.s.l., average slope 9.0°, average TWI 7.0, average NDVI 0.53). Key limitations: hilly morphology with significant slopes (52% over 8°); rapid drainage and vulnerability to summer drought (31% with low TWI or south-facing slopes).


,posizione,crop_code,crop_name,category,adattabilita_pct,punteggio_finale,note_agronomiche
0,1,PST,Pasture grasses,Fodder crops,64.4,70.1,Favored for soil protection
1,2,BRL,Barley,Cereals,53.4,63.0,Favored for soil protection; favored for droug...
2,3,OAT,Oat,Cereals,53.4,59.1,Favored for soil protection
3,4,WHE,Wheat,Cereals,53.4,59.1,Favored for soil protection
4,5,RYE,Rye,Cereals,52.7,58.4,Favored for soil protection
...,...,...,...,...,...,...,...
72,73,TANN,Tannia (Xanthosoma sagittifolium),Roots and tubers,0.0,0.0,Standard climatic adaptability
73,74,SUC,Sugar cane,Sugar crops,0.0,0.0,Standard climatic adaptability
74,75,TAROW,Taro wetland (Colocasia esculenta),Roots and tubers,0.0,0.0,Standard climatic adaptability
75,76,TAROD,Taro dryland (Colocasia esculenta),Roots and tubers,0.0,0.0,Standard climatic adaptability


In [ ]:
from IPython.display import FileLink

nome_file = "risultato_asciano"
risultato.to_csv(nome_file, index=False)

FileLink(nome_file)

/kaggle/working/risultato_asciano

In [ ]:
# mugello
bbox_utente = (11.30, 43.92, 11.50, 44.06)
risultato_mugello = analizza_zona_geografica(df_terreno, df_crops, bbox=bbox_utente)
display(risultato_mugello[["posizione", "crop_code", "crop_name", "category", "adattabilita_pct", "punteggio_finale", "note_agronomiche"]])

=== TERRAIN DESCRIPTION ===
Analyzed area: 21795.3 ha (average elevation -4569 m a.s.l., average slope -4798.6°, average TWI 7.1, average NDVI 0.00). Key limitations: hilly morphology with significant slopes (36% over 8°); rapid drainage and vulnerability to summer drought (23% with low TWI or south-facing slopes).


,posizione,crop_code,crop_name,category,adattabilita_pct,punteggio_finale,note_agronomiche
0,1,PST,Pasture grasses,Fodder crops,26.1,29.7,Favored for soil protection
1,2,BRL,Barley,Cereals,19.1,25.6,Favored for soil protection; favored for droug...
2,3,TRI,Triticale,Cereals,19.5,23.1,Favored for soil protection
3,4,WHE,Wheat,Cereals,19.5,23.1,Favored for soil protection
4,5,RYE,Rye,Cereals,19.5,23.1,Favored for soil protection
...,...,...,...,...,...,...,...
72,73,SUC,Sugar cane,Sugar crops,0.0,0.0,Standard climatic adaptability
73,74,TAROW,Taro wetland (Colocasia esculenta),Roots and tubers,0.0,0.0,Standard climatic adaptability
74,75,TAROD,Taro dryland (Colocasia esculenta),Roots and tubers,0.0,0.0,Standard climatic adaptability
75,76,TANN,Tannia (Xanthosoma sagittifolium),Roots and tubers,0.0,0.0,Standard climatic adaptability


In [ ]:
from IPython.display import FileLink

nome_file = "risultato_mugello"
risultato.to_csv(nome_file, index=False)

FileLink(nome_file)

/kaggle/working/risultato_mugello

In [ ]:
# val di chiana
bbox_utente = (11.76, 43.20, 11.96, 43.34)
risultato_chiana = analizza_zona_geografica(df_terreno, df_crops, bbox=bbox_utente)
display(risultato_chiana[["posizione", "crop_code", "crop_name", "category", "adattabilita_pct", "punteggio_finale", "note_agronomiche"]])

=== TERRAIN DESCRIPTION ===
Analyzed area: 33366.3 ha (average elevation 268 m a.s.l., average slope 3.6°, average TWI 8.3, average NDVI 0.36). Morphologically balanced terrain, with gentle slopes, good drainage and no significant physical limitations.


,posizione,crop_code,crop_name,category,adattabilita_pct,punteggio_finale,note_agronomiche
0,1,CAM,Camelina,Oil crops,57.2,57.2,Standard climatic adaptability
1,2,WHE,Wheat,Cereals,55.9,55.9,Standard climatic adaptability
2,3,BRL,Barley,Cereals,54.0,54.0,Standard climatic adaptability
3,4,BCK,Buckwheat,Cereals,52.9,52.9,Standard climatic adaptability
4,5,OAT,Oat,Cereals,50.5,50.5,Standard climatic adaptability
...,...,...,...,...,...,...,...
72,73,SUC,Sugar cane,Sugar crops,0.0,0.0,Standard climatic adaptability
73,74,TAROW,Taro wetland (Colocasia esculenta),Roots and tubers,0.0,0.0,Standard climatic adaptability
74,75,TAROD,Taro dryland (Colocasia esculenta),Roots and tubers,0.0,0.0,Standard climatic adaptability
75,76,TANN,Tannia (Xanthosoma sagittifolium),Roots and tubers,0.0,0.0,Standard climatic adaptability


In [ ]:
from IPython.display import FileLink

nome_file = "risultato_chiana"
risultato.to_csv(nome_file, index=False)

FileLink(nome_file)

/kaggle/working/risultato_chiana